In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
%%capture
!pip install bert-score pyarrow

## **01. Config**

In [3]:
from pathlib import Path
import importlib
import itertools
import json
import re
import sys
import time

import numpy as np
import pandas as pd
import torch

In [4]:
BASE_DIR = Path("/content/drive/MyDrive/Legal_Sytem")
DATA_DIR = BASE_DIR / "01_Dataset/02_processed_data"
RETRIEVER_DIR = BASE_DIR / "02_Retriever"
GENERATOR_DIR = BASE_DIR / "03_Generator"

BM25_OUTPUT_DIR = RETRIEVER_DIR / "02_Baseline/BM25/output"
DENSE_OUTPUT_DIR = RETRIEVER_DIR / "03_Finetuning/output"
HYBRID_OUTPUT_DIR = RETRIEVER_DIR / "04_Hybrid/output"

MODELS_DIR = GENERATOR_DIR / "models"
OUTPUT_DIR = BASE_DIR / "04_End2End/output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
EVALUATION_SPLIT = 'test'

RETRIEVER_RANKING_PATHS = {
    'BM25': BM25_OUTPUT_DIR / f'{EVALUATION_SPLIT}_ranking.parquet',
    'multilingual-e5 (dense)': DENSE_OUTPUT_DIR / f'multilingual-e5-large__marginmse-cross_encoder__{EVALUATION_SPLIT}_ranking.parquet',
    'Hybrid': HYBRID_OUTPUT_DIR / f'best_hybrid_ranking.parquet'
}

In [6]:
GENERATOR_RUN_NAMES = ['Qwen3-0.6B', 'gemma-3-270m-it', 'Qwen3-1.7B']

RETRIEVAL_TOP_K = 1

In [7]:
GENERATION_BATCH_SIZE = 32
USE_BERTSCORE = True
BERTSCORE_MODEL = None
TOKENIZE_KWARGS = dict(lowercase=True, strip_punctuation=True)
RANDOM_SEED = 42

In [9]:
sys.path.append(str(GENERATOR_DIR))
sys.path.append(str(RETRIEVER_DIR))

import generator_common
import retriever_common

importlib.reload(generator_common)
importlib.reload(retriever_common)

gc = generator_common
rc = retriever_common

gc.require_version(5)
rc.require_version(3)

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

generator_common v5 (need >= 5)
retriever_common v3 (cần >= 3)
Device: cuda


## **02. Import data**

In [10]:
qa_master = pd.read_parquet(DATA_DIR / 'corpus' / 'qa_master.parquet')
corpus = pd.read_parquet(DATA_DIR / 'corpus' / 'corpus.parquet')
qrels = pd.read_parquet(DATA_DIR / 'corpus' / 'qrels.parquet')

rankings = {}
missing_rankings = []

for retriever_name, path in RETRIEVER_RANKING_PATHS.items():
  if not path.exists():
    missing_rankings.append((retriever_name, path))
    continue

  ranking = pd.read_parquet(path)

  assert {'query_id', 'doc_id', 'rank'} <= set(ranking.columns), (f"{retriever_name}: đây không phải file ranking. Cột: {list(ranking.columns)}")

  rankings[retriever_name] = ranking

  print(f"{retriever_name:18s}: {len(ranking):,} dòng, {ranking['query_id'].nunique():,} câu hỏi, độ sâu {ranking['rank'].max()}")

BM25              : 47,900 dòng, 958 câu hỏi, độ sâu 50
multilingual-e5 (dense): 47,900 dòng, 958 câu hỏi, độ sâu 50
Hybrid            : 47,900 dòng, 958 câu hỏi, độ sâu 50


In [11]:
for retriever_name, path in missing_rankings:
  print(f"THIẾU {retriever_name}: {path}")

if any(name == 'Hybrid' for name, _ in missing_rankings):
  print("\n=> Notebook 04_hybrid chưa lưu ranking ra parquet. Thêm cell rc.save_run(...) ở cuối notebook đó rồi chạy lại.")

assert rankings, "Không nạp được retriever nào."

## **03. Retriever index**

In [12]:
qrels_map = rc.build_qrels_map(qrels[qrels['split'] == EVALUATION_SPLIT])

retriever_records = []

for retriever_name, ranking in rankings.items():
  summary = rc.summarize(rc.evaluation(ranking, qrels_map))

  retriever_records.append({
      'retriever': retriever_name,
      **{f'p@{k}': round(summary[f'p@{k}'] * 100, 2) for k in [1, 5, 10, 15, 20]},
      **{f'mrr@{k}': round(summary[f'mrr@{k}'] * 100, 2) for k in [1, 5, 10, 15, 20]},
      **{f'ndcg@{k}': round(summary[f'ndcg@{k}'] * 100, 2) for k in [1, 5, 10, 15, 20]},
      **{f'hit@{k}': round(summary[f'hit@{k}'] * 100, 2) for k in [1, 5, 10, 15, 20]}
  })

retriever_table = pd.DataFrame(retriever_records)

display(retriever_table)

,retriever,p@1,p@5,p@10,p@15,p@20,mrr@1,mrr@5,mrr@10,mrr@15,...,ndcg@1,ndcg@5,ndcg@10,ndcg@15,ndcg@20,hit@1,hit@5,hit@10,hit@15,hit@20
0,BM25,66.91,89.04,92.90,94.78,95.62,66.91,76.18,76.71,76.87,...,66.91,79.44,80.71,81.21,81.41,66.91,89.04,92.90,94.78,95.62
1,multilingual-e5 (dense),73.38,92.07,95.41,96.24,96.87,73.38,81.17,81.65,81.71,...,73.38,83.93,85.05,85.27,85.41,73.38,92.07,95.41,96.24,96.87
2,Hybrid,81.52,96.45,97.91,98.54,98.75,81.52,87.41,87.61,87.66,...,81.52,89.68,90.16,90.32,90.37,81.52,96.45,97.91,98.54,98.75


In [13]:
retriever_hit_at_k = {
    row['retriever']: row[f'hit@{RETRIEVAL_TOP_K}'] / 100 for row in retriever_records
}

In [14]:
# retriever_hit_at_k = {}
# for row in retriever_records:
#   retriever_name = row['retriever']
#   retriever_hit_at_k[retriever_name] = {}
#   for k in RETRIEVAL_TOP_K:
#     retriever_hit_at_k[retriever_name][f'hit@{k}'] = row[f'hit@{k}'] / 100
# display(retriever_hit_at_k)

## **04. Generator**

In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM

class LoadedGenerator:
  def __init__(self, run_name: str):
    self.run_name = run_name
    self.model_dir = MODELS_DIR / run_name

    config_path = self.model_dir / 'prompt_config.json'

    if not config_path.exists():
      raise FileNotFoundError(f"Thiếu {config_path}. Checkpoint này được lưu bởi notebook 03 bản cũ — chạy lại mục 08 của 03__Generator_train để sinh prompt_config.json, hoặc dựng lại file bằng tay đúng như lúc fine-tune.")

    with open(config_path, encoding='utf-8') as file:
      self.config = json.load(file)

    self.tokenizer = AutoTokenizer.from_pretrained(str(self.model_dir), trust_remote_code=True)

    if self.tokenizer.pad_token is None:
      self.tokenizer.pad_token = self.tokenizer.eos_token

    self.model = AutoModelForCausalLM.from_pretrained(str(self.model_dir), torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32, trust_remote_code=True)

    self.model = self.model.to('cuda' if torch.cuda.is_available() else 'cpu')
    self.model.eval()


  def render_prompt(self, question: str, context: str) -> str:
    user_message = self.config['user_template'].format(question=str(question).strip(), context=str(context).strip())

    messages = [
        {'role': 'system', 'content': self.config['system_prompt']},
        {'role': 'user', 'content': user_message},
    ]

    template_kwargs = dict(tokenize=False, add_generation_prompt=True)

    if self.config.get('enable_thinking') is not None:
      template_kwargs['enable_thinking'] = self.config['enable_thinking']

    try:
      return self.tokenizer.apply_chat_template(messages, **template_kwargs)
    except Exception:
      merged = [{'role': 'user',
                 'content': self.config['system_prompt'] + "\n\n" + user_message}]

      return self.tokenizer.apply_chat_template(merged, **template_kwargs)

  @staticmethod
  def clean_generation(text: str) -> str:
    text = re.sub(r'<think>.*?</think>', ' ', text, flags=re.DOTALL)

    return re.sub(r'\s+', ' ', text).strip()

  @torch.no_grad()
  def generate(self, frame: pd.DataFrame, batch_size: int = GENERATION_BATCH_SIZE) -> list[str]:
    original_padding_side = self.tokenizer.padding_side
    original_truncation_side = self.tokenizer.truncation_side

    self.tokenizer.padding_side = 'left'
    self.tokenizer.truncation_side = 'left'   # khớp cách cắt lúc huấn luyện

    prompts = [
        self.render_prompt(question, context)
        for question, context in zip(frame['source_question'], frame['source_context'])
    ]

    outputs = []

    for start in range(0, len(prompts), batch_size):
      encoded = self.tokenizer(prompts[start:start + batch_size], return_tensors='pt', padding=True, truncation=True, max_length=self.config['max_source_length'], add_special_tokens=False)

      encoded = {key: value.to(self.model.device) for key, value in encoded.items()}

      generated = self.model.generate(**encoded, max_new_tokens=self.config['max_target_length'], num_beams=self.config.get('num_beams', 1), do_sample=self.config.get('do_sample', False), pad_token_id=self.tokenizer.pad_token_id, eos_token_id=self.tokenizer.eos_token_id)
      new_tokens = generated[:, encoded['input_ids'].shape[1]:]

      outputs.extend(self.tokenizer.batch_decode(new_tokens, skip_special_tokens=True))

    self.tokenizer.padding_side = original_padding_side
    self.tokenizer.truncation_side = original_truncation_side

    return [self.clean_generation(text) for text in outputs]

  def free(self):
    del self.model
    torch.cuda.empty_cache()


## **05. Nạp model và thực thiện generate**

In [ ]:
result_rows = []
per_example_store = {}
prediction_store = {}

for generator_run_name in GENERATOR_RUN_NAMES:
  if not (MODELS_DIR / generator_run_name).exists():
    print(f"Bỏ qua '{generator_run_name}': không có thư mục checkpoint.")
    continue

  generator = LoadedGenerator(generator_run_name)

  passage_column = generator.config['passage_column']
  target_column = generator.config['target_column']

  print(f"\n=== {generator_run_name} "
        f"(passage_column={passage_column}, target={target_column}) ===")

  gold_frame = gc.build_generator_dataframe(qa_master, EVALUATION_SPLIT, corpus, passage_column=passage_column, target_column=target_column)

  references = gold_frame['target'].tolist()

  start_time = time.perf_counter()

  oracle_hypotheses = generator.generate(gold_frame)

  oracle_seconds = time.perf_counter() - start_time

  oracle_per_example = gc.evaluate_generations(references, oracle_hypotheses, gold_frame['qa_id'].tolist(), use_bertscore=USE_BERTSCORE, bertscore_model=BERTSCORE_MODEL, **TOKENIZE_KWARGS)

  oracle_summary = gc.summarize_generations(references, oracle_hypotheses, oracle_per_example, **TOKENIZE_KWARGS)

  result_rows.append({
      'generator': generator_run_name, 'retriever': '— (oracle)',
      'nhóm': 'Oracle (gold context)', 'n': len(gold_frame),
      'hit@k': None, **oracle_summary,
      'latency_ms': round(oracle_seconds / len(gold_frame) * 1000, 2),
  })

  per_example_store[(generator_run_name, 'oracle')] = oracle_per_example
  prediction_store[(generator_run_name, 'oracle')] = gold_frame.assign(prediction=oracle_hypotheses)

  print(f"  Oracle: " + "  ".join(f"{k}={v}" for k, v in oracle_summary.items()))

  for retriever_name, ranking in rankings.items():
    end_to_end_frame = gc.attach_retrieved_context(gold_frame, ranking, corpus, top_k=RETRIEVAL_TOP_K, passage_column=passage_column,context_separator=generator.config['context_separator'])

    start_time = time.perf_counter()

    hypotheses = generator.generate(end_to_end_frame)

    generation_seconds = time.perf_counter() - start_time

    per_example = gc.evaluate_generations(references, hypotheses, end_to_end_frame['qa_id'].tolist(), use_bertscore=USE_BERTSCORE, bertscore_model=BERTSCORE_MODEL, **TOKENIZE_KWARGS)

    per_example['retrieved_hit'] = end_to_end_frame['retrieved_hit'].to_numpy()

    per_example_store[(generator_run_name, retriever_name)] = per_example
    prediction_store[(generator_run_name, retriever_name)] = end_to_end_frame.assign(prediction=hypotheses)

    hit_rate = float(end_to_end_frame['retrieved_hit'].mean())

    result_rows.append({
        'generator': generator_run_name, 'retriever': retriever_name,
        'nhóm': f'End-to-end top-{RETRIEVAL_TOP_K}', 'n': len(end_to_end_frame),
        'hit@k': round(hit_rate * 100, 2),
        **gc.summarize_generations(references, hypotheses, per_example, **TOKENIZE_KWARGS),
        'latency_ms': round(generation_seconds / len(end_to_end_frame) * 1000, 2),
    })

    # Tách theo retriever đúng / sai
    for label, mask in [('├ retriever ĐÚNG', per_example['retrieved_hit']),
                        ('└ retriever SAI', ~per_example['retrieved_hit'])]:
      subset = per_example[mask]

      if subset.empty:
        continue

      indices = subset.index.to_numpy()

      result_rows.append({
          'generator': generator_run_name, 'retriever': retriever_name,
          'nhóm': label, 'n': len(subset), 'hit@k': None,
          **gc.summarize_generations(
              [references[i] for i in indices],
              [hypotheses[i] for i in indices],
              subset, **TOKENIZE_KWARGS),
      })

    print(f"  {retriever_name:18s}: hit@{RETRIEVAL_TOP_K}={hit_rate * 100:.2f}%  "
          f"BLEU1={result_rows[-3]['BLEU1']}")

  generator.free()

result_table = pd.DataFrame(result_rows)

display(result_table)

result_table.to_csv(OUTPUT_DIR / f'end2end_matrix__{EVALUATION_SPLIT}.csv', index=False, encoding='utf-8-sig')


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]


=== Qwen3-0.6B (passage_column=embedding_text, target=abstractive answer) ===


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

... (40 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Hybrid            : hit@1=81.52%  BLEU1=70.99


,generator,retriever,nhóm,n,hit@k,BLEU1,BLEU4,ROUGE-L,F1,BERTScore,latency_ms
0,Qwen3-0.6B,— (oracle),Oracle (gold context),958,NaN,74.60,67.50,74.11,78.81,90.949997,502.49
1,Qwen3-0.6B,BM25,End-to-end top-1,958,66.91,62.76,54.09,65.08,70.17,87.900002,559.45
2,Qwen3-0.6B,BM25,├ retriever ĐÚNG,641,NaN,72.98,65.93,75.51,80.61,91.480003,NaN
3,Qwen3-0.6B,BM25,└ retriever SAI,317,NaN,36.80,25.24,43.99,49.06,80.669998,NaN
4,Qwen3-0.6B,multilingual-e5 (dense),End-to-end top-1,958,73.38,65.16,57.33,67.88,72.65,88.849998,577.95
5,Qwen3-0.6B,multilingual-e5 (dense),├ retriever ĐÚNG,703,NaN,74.08,67.37,75.11,79.94,91.339996,NaN
6,Qwen3-0.6B,multilingual-e5 (dense),└ retriever SAI,255,NaN,39.73,28.32,47.94,52.56,81.989998,NaN
7,Qwen3-0.6B,Hybrid,End-to-end top-1,958,81.52,67.76,59.95,69.76,74.76,89.500000,561.62
8,Qwen3-0.6B,Hybrid,├ retriever ĐÚNG,781,NaN,74.20,67.32,75.04,79.93,91.349998,NaN
9,Qwen3-0.6B,Hybrid,└ retriever SAI,177,NaN,39.60,27.32,46.47,51.92,81.320000,NaN


In [ ]:
METRIC = 'BLEU1'

decomposition_records = []

for generator_run_name in result_table['generator'].unique():
  subset = result_table[result_table['generator'] == generator_run_name]

  oracle_score = subset.loc[subset['nhóm'] == 'Oracle (gold context)', METRIC].iloc[0]

  for retriever_name in rankings:
    end_to_end = subset[(subset['retriever'] == retriever_name) & (subset['nhóm'].str.startswith('End-to-end'))]

    correct = subset[(subset['retriever'] == retriever_name) & (subset['nhóm'] == '├ retriever ĐÚNG')]

    if end_to_end.empty:
      continue

    end_to_end_score = end_to_end[METRIC].iloc[0]
    correct_score = correct[METRIC].iloc[0] if not correct.empty else np.nan
    hit_rate = end_to_end['hit@k'].iloc[0] / 100

    decomposition_records.append({
        'generator': generator_run_name,
        'retriever': retriever_name,
        f'oracle_{METRIC}': oracle_score,
        f'end2end_{METRIC}': end_to_end_score,
        'tổng_mất': round(oracle_score - end_to_end_score, 2),
        'hit@k': round(hit_rate * 100, 2),
        'do_retriever': round((1 - hit_rate) * oracle_score, 2),
   
        'do_generator_khi_context_đúng': round(oracle_score - correct_score, 2),
    })

decomposition_table = pd.DataFrame(decomposition_records)

display(decomposition_table)

decomposition_table.to_csv(OUTPUT_DIR / f'error_decomposition__{EVALUATION_SPLIT}.csv', index=False, encoding='utf-8-sig')

print("\nĐọc bảng: 'do_generator_khi_context_đúng' lẽ ra phải gần 0 — cùng câu hỏi, cùng điều luật, chỉ khác đường lấy văn bản. Lệch quá 1-2 điểm nghĩa là passage_column hoặc cách ghép context ở hai đường không giống nhau.")


,generator,retriever,oracle_BLEU1,end2end_BLEU1,tổng_mất,hit@k,do_retriever,do_generator_khi_context_đúng
0,Qwen3-0.6B,BM25,74.60,62.76,11.84,66.91,24.69,1.62
1,Qwen3-0.6B,multilingual-e5 (dense),74.60,65.16,9.44,73.38,19.86,0.52
2,Qwen3-0.6B,Hybrid,74.60,67.76,6.84,81.52,13.79,0.40
3,gemma-3-270m-it,BM25,62.24,51.50,10.74,66.91,20.60,-3.31
4,gemma-3-270m-it,multilingual-e5 (dense),62.24,56.75,5.49,73.38,16.57,-2.81
5,gemma-3-270m-it,Hybrid,62.24,58.41,3.83,81.52,11.50,-2.99
6,Qwen3-1.7B,BM25,76.90,63.16,13.74,66.91,25.45,0.10
7,Qwen3-1.7B,multilingual-e5 (dense),76.90,69.08,7.82,73.38,20.47,-0.39
8,Qwen3-1.7B,Hybrid,76.90,70.99,5.91,81.52,14.21,-0.66



Đọc bảng: 'do_generator_khi_context_đúng' lẽ ra phải gần 0 — cùng câu hỏi, cùng điều luật, chỉ khác đường lấy văn bản. Lệch quá 1-2 điểm nghĩa là passage_column hoặc cách ghép context ở hai đường không giống nhau.


## **07. Phân tích hành vi trả lời**

In [18]:
behaviour_records = []

for (generator_run_name, retriever_name), predictions in prediction_store.items():
  if retriever_name == 'oracle':
    continue

  report = gc.behaviour_report(predictions['prediction'].tolist(), predictions['retrieved_hit'].tolist())

  report.insert(0, 'generator', generator_run_name)
  report.insert(1, 'retriever', retriever_name)

  behaviour_records.append(report)

behaviour_table = pd.concat(behaviour_records, ignore_index=True)

display(behaviour_table)

behaviour_table.to_csv(OUTPUT_DIR / f'behaviour__{EVALUATION_SPLIT}.csv', index=False, encoding='utf-8-sig')

print("\n'khẳng_định_mà_không_từ_chối' trên nhóm Retriever SAI là chỉ số rủi ro: model được đưa nhầm điều luật, không nói gì về việc thiếu thông tin, mà vẫn đưa ra con số/ngưỡng cụ thể.")

,generator,retriever,nhóm,n,tỷ_lệ_từ_chối,tỷ_lệ_khẳng_định_cụ_thể,khẳng_định_mà_không_từ_chối
0,Qwen3-0.6B,BM25,Retriever ĐÚNG,641,2.65,9.83,9.20
1,Qwen3-0.6B,BM25,Retriever SAI,317,0.63,9.15,9.15
2,Qwen3-0.6B,multilingual-e5 (dense),Retriever ĐÚNG,703,2.28,9.96,9.39
3,Qwen3-0.6B,multilingual-e5 (dense),Retriever SAI,255,0.39,8.24,8.24
4,Qwen3-0.6B,Hybrid,Retriever ĐÚNG,781,2.18,9.22,8.71
5,Qwen3-0.6B,Hybrid,Retriever SAI,177,0.56,6.78,6.78
6,gemma-3-270m-it,BM25,Retriever ĐÚNG,641,2.50,9.52,9.05
7,gemma-3-270m-it,BM25,Retriever SAI,317,1.58,9.78,9.46
8,gemma-3-270m-it,multilingual-e5 (dense),Retriever ĐÚNG,703,2.56,9.53,9.10
9,gemma-3-270m-it,multilingual-e5 (dense),Retriever SAI,255,1.18,8.24,7.84



'khẳng_định_mà_không_từ_chối' trên nhóm Retriever SAI là chỉ số rủi ro: model được đưa nhầm điều luật, không nói gì về việc thiếu thông tin, mà vẫn đưa ra con số/ngưỡng cụ thể.


## 08 — Kiểm định thống kê

In [ ]:
BOOTSTRAP_METRIC = 'rouge_l'

significance_records = []

best_retriever = max(retriever_hit_at_k, key=retriever_hit_at_k.get)

available_generators = sorted({generator_name for generator_name, retriever_name in per_example_store if retriever_name == best_retriever})

for name_a, name_b in itertools.combinations(available_generators, 2):
  result = gc.paired_bootstrap(per_example_store[(name_a, best_retriever)], per_example_store[(name_b, best_retriever)], metric=BOOTSTRAP_METRIC)

  significance_records.append({
      'so_sánh': f'{name_a} vs {name_b}', 'điều_kiện': f'end-to-end / {best_retriever}',
      'A': round(result['mean_a'] * 100, 2), 'B': round(result['mean_b'] * 100, 2),
      'chênh_lệch': round(result['difference'] * 100, 2),
      'ci_95': f"[{result['ci_lower_95'] * 100:+.2f}, {result['ci_upper_95'] * 100:+.2f}]",
      'p_value': round(result['p_value'], 4),
      'significant': result['significant_at_0.05'],
  })



In [ ]:
if available_generators:
  best_generator = available_generators[0]

  for name_a, name_b in itertools.combinations(rankings, 2):
    key_a = (best_generator, name_a)
    key_b = (best_generator, name_b)

    if key_a not in per_example_store or key_b not in per_example_store:
      continue

    result = gc.paired_bootstrap(per_example_store[key_a], per_example_store[key_b], metric=BOOTSTRAP_METRIC)

    significance_records.append({
        'so_sánh': f'{name_a} vs {name_b}', 'điều_kiện': f'generator = {best_generator}',
        'A': round(result['mean_a'] * 100, 2), 'B': round(result['mean_b'] * 100, 2),
        'chênh_lệch': round(result['difference'] * 100, 2),
        'ci_95': f"[{result['ci_lower_95'] * 100:+.2f}, {result['ci_upper_95'] * 100:+.2f}]",
        'p_value': round(result['p_value'], 4),
        'significant': result['significant_at_0.05'],
    })

significance_table = pd.DataFrame(significance_records)

display(significance_table)

significance_table.to_csv(OUTPUT_DIR / f'significance__{EVALUATION_SPLIT}.csv', index=False, encoding='utf-8-sig')

,so_sánh,điều_kiện,A,B,chênh_lệch,ci_95,p_value,significant
0,Qwen3-0.6B vs Qwen3-1.7B,end-to-end / Hybrid,69.76,70.29,-0.53,"[-1.72, +0.64]",0.3747,False
1,Qwen3-0.6B vs gemma-3-270m-it,end-to-end / Hybrid,69.76,61.96,7.80,"[+6.12, +9.50]",0.0000,True
2,Qwen3-1.7B vs gemma-3-270m-it,end-to-end / Hybrid,70.29,61.96,8.33,"[+6.54, +10.14]",0.0000,True
3,BM25 vs multilingual-e5 (dense),generator = Qwen3-0.6B,65.08,67.88,-2.80,"[-4.10, -1.51]",0.0000,True
4,BM25 vs Hybrid,generator = Qwen3-0.6B,65.08,69.76,-4.68,"[-5.82, -3.58]",0.0000,True
5,multilingual-e5 (dense) vs Hybrid,generator = Qwen3-0.6B,67.88,69.76,-1.88,"[-2.73, -1.04]",0.0000,True


## 09 — Lưu dự đoán để phân tích lỗi bằng mắt

In [ ]:
for (generator_run_name, retriever_name), predictions in prediction_store.items():
  safe_name = retriever_name.replace(' ', '_').replace('(', '').replace(')', '')

  predictions.to_parquet(OUTPUT_DIR / f'{generator_run_name}__{safe_name}__{EVALUATION_SPLIT}_predictions.parquet', index=False)

print("Đã lưu vào:", OUTPUT_DIR)

sample_key = (available_generators[0], best_retriever) if available_generators else None

if sample_key and sample_key in prediction_store:
  wrong = prediction_store[sample_key]
  wrong = wrong[~wrong['retrieved_hit']].sample(min(5, int((~prediction_store[sample_key]['retrieved_hit']).sum())), random_state=RANDOM_SEED)

  for row in wrong.itertuples(index=False):
    print("Câu hỏi        :", row.source_question)
    print("Điều luật ĐÚNG :", row.gold_context[:150], "...")
    print("Điều luật LẤY  :", row.source_context[:150], "...")
    print("Model trả lời  :", row.prediction)
    print("Đáp án chuẩn   :", row.target)
    print("-" * 90)

Đã lưu vào: /content/drive/MyDrive/Legal_Sytem/04_End2End/output
Câu hỏi        : Đơn vị nào có trách nhiệm chuẩn bị hồ sơ?
Điều luật ĐÚNG : Văn bản: QUY CHẾ Văn bằng, chứng chỉ của Trường Đại học Công nghệ Thông tin
Điều  10. Trách nhiệm in, quản lý phôi văn bằng, chứng chỉ
1. Trường ĐHCNT ...
Điều luật LẤY  : Văn bản: QUY ĐỊNH Tổ chức thi các môn học hệ đại học chính quy của Trường Đại học Công nghệ Thông tin
Điều  11. Quy trình chuẩn bị cho kỳ thi
1. Đầu v ...
Model trả lời  : Phòng Đào tạo Đại học/ Văn phòng Đặc biệt có trách nhiệm chuẩn bị hồ sơ.
Đáp án chuẩn   : Trách nhiệm chuẩn bị hồ sơ là của PĐTĐH và PĐTSĐH-KHCN.
------------------------------------------------------------------------------------------
Câu hỏi        : Yêu cầu về chứng chỉ tiếng Nhật như thế nào đối với sinh viên chương trình liên thông?
Điều luật ĐÚNG : Văn bản: QUY ĐỊNH ĐÀO TẠO NGOẠI NGỮ ĐỐI VỚI HỆ ĐẠI HỌC CHÍNH QUY CỦA TRƯỜNG ĐẠI HỌC CÔNG NGHỆ THÔNG TIN
Điều  9.	Văn bằng, chứng chỉ/chứng nhận được s ...
Đi